In [ ]:
!git clone https://github.com/Motoki0705/tennis-lab
%pip install pytorch-lightning hydra-core
from google.colab import drive
drive.mount('/content/drive')
%cd /content/tennis-lab
!cp -av /content/drive/MyDrive/tennis_lab/data/court.tar.zst /content/
!apt-get update
!apt-get install -y zstd
!mkdir -p /content/tennis-lab/data
!tar -I zstd -xf /content/drive_upload_archives/court.tar.zst -C /content/tennis-lab/data

In [ ]:
!git submodule update --init --recursive third_party/DINO
%cd /content/tennis-lab

!mkdir -p /content/tennis-lab/checkpoints/DINO

!python /content/tennis-lab/checkpoints/DINO/scripts/download_checkpoint0027_5scale_swin.py \
  --output /content/tennis-lab/checkpoints/DINO/checkpoint0027_5scale_swin.pth

!python /content/tennis-lab/checkpoints/DINO/scripts/load_dino_swin_backbone.py \
  --checkpoint /content/tennis-lab/checkpoints/DINO/checkpoint0027_5scale_swin.pth \
  --save-backbone-state /content/tennis-lab/checkpoints/DINO/swin_backbone_state_checkpoint0027_5scale.pth \
  --save-full-backbone-module /content/tennis-lab/checkpoints/DINO/dino_swin_backbone_module_state_checkpoint0027_5scale.pth \
  --strict

!python /content/tennis-lab/checkpoints/DINO/scripts/inspect_dino_swin_backbone_output.py \
  --checkpoint /content/tennis-lab/checkpoints/DINO/swin_backbone_state_checkpoint0027_5scale.pth \
  --use-checkpoint \
  --device cuda \
  --height 384 \
  --width 384 \
  --strict

In [ ]:
!python -m src.tasks.court_detection.scripts.train run.output_dir="/content/drive/MyDrive/tennis_lab/outputs/court_detection" model=court_seg_dino_swin_fpn model.name=court_hierarchical_dino_swin_fpn_kp20 model.num_classes=20 data=court_kp training.trainer.check_val_every_n_epoch=5 training.qualitative_logging.every_n_epochs=5 data.batch_size=16